# Part Y: Structural identifiabiltiy analysis

Structural identifiabiltiy analysis aims to describe what parameter values (and other model properties) are fundamentally possible to determine uniquely from a specific type of data. It assumes that infinite quantitites of noisless data is avaiable, and is different from *practical identifiabiltiy analysis* which aims to describe what model properties can be determiend given *avaiable* data. In practise, automated software tools, like the [StructuralIdentifiability.jl](https://github.com/SciML/StructuralIdentifiability.jl) package, is used to determine structural identifiability.

We first consider the simple rabiit exponential growth model. We declare it as a reaction system with a single reaction (a rabbit giving birth at rate *bf\*bn*).

In [1]:
using Catalyst 
rn = @reaction_network begin
    bf*bn, R --> 2R
end

Model ##ReactionSystem#285:
Unknowns (1): see unknowns(##ReactionSystem#285)
  R(t)
Parameters (2): see parameters(##ReactionSystem#285)
  bf
  bn

We can check the corresponding ODE model:

In [2]:
ode_model(rn)

Model ##ReactionSystem#285:
Equations (1):
  1 standard: see equations(##ReactionSystem#285)
Unknowns (1): see unknowns(##ReactionSystem#285)
  R(t)
Parameters (2): see parameters(##ReactionSystem#285)
  bf
  bn

Finally, we can determine structural identifiabiltiy using the `assess_identifiability` function. here, we also designate what quantities we can meassure.

In [3]:
using StructuralIdentifiability
assess_identifiability(rn; measured_quantities = [:R])

┌ Info: System parsed into R' = R*bf*bn
└ (___internal_observables(get_iv))[1] = R
[ Info: Assessing local identifiability
[ Info: Assessing global identifiability
[ Info: Functions to check involve states
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 5.113542902 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 0.695783319 seconds
[ Info: Dimensions of the Wronskians [2]
[ Info: Ranks of the Wronskians computed in 0.021508144 seconds
[ Info: Simplifying generating set. Simplification level: standard
✓ # Computing specializations..     Time: 0:00:04
[ Info: Search for polynomial generators concluded in 3.89247386
[ Info: Selecting generators in 0.006287543
[ Info: Inclusion checked with probability 0.9955 in 6.101669913 seconds
[ Info: Global identifiability assessed in 38.638598527 seconds


OrderedCollections.OrderedDict{SymbolicUtils.BasicSymbolicImpl.var"typeof(BasicSymbolicImpl)"{SymReal}, Symbol} with 3 entries:
  R(t) => :globally
  bf   => :nonidentifiable
  bn   => :nonidentifiable

This case was trivial, however, StructuralIdentifiabiltiy.jl can also deal with much more compcliaetd models. Let's now the Goodwin oscilaltor model:

In [4]:
gwo = @reaction_network begin
    (pₘ/(1+P), dₘ), 0 <--> M
    (pₑ*M,dₑ), 0 <--> E
    (pₚ*E,dₚ), 0 <--> P
end
ode_model(gwo)

Model ##ReactionSystem#294:
Equations (3):
  3 standard: see equations(##ReactionSystem#294)
Unknowns (3): see unknowns(##ReactionSystem#294)
  M(t)
  E(t)
  P(t)
Parameters (6): see parameters(##ReactionSystem#294)
  pₘ
  dₘ
  pₑ
  dₑ
  ⋮

Here, we see that we have acombination of *globally identifiable*, *localy identifiable*, and *non-identifiable* parameters.

In [ ]:
assess_identifiability(gwo; measured_quantities = [:M])

┌ Info: System parsed into M' = (-M*P*dₘ - M*dₘ + pₘ)//(P + 1)
│ E' = M*pₑ - E*dₑ
│ P' = E*pₚ - P*dₚ
└ (___internal_observables(get_iv))[1] = M
[ Info: Assessing local identifiability
[ Info: Assessing global identifiability
[ Info: Functions to check involve states
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 0.449257385 seconds


A locally identifiable parameter is a parameter such that one can determine it down to a countable number of potential values, but not to a single value. I.e. in a model where $K^2$ appears, $K$ and $-K$ yileds identical dynamics, and $K$ is thus locally identifiable. One one only cares about local identifiability, the `assess_local_identifiability` function can be used (which generally computes quicker as comapred to `assess_identifiability`).

In [ ]:
assess_local_identifiability(gwo; measured_quantities = [:M])

Finally, we can use the `find_identifiable_functions` function to find a list of all possible expressions that we *can* identify:

In [ ]:
find_identifiable_functions(gwo; measured_quantities = [:M])

Here, we can see that the identifiable parameters appear as identifiable expressions, while the nin-identifiable ones are entangled with other parameters.

We can also list multiple measured quantities in `measured_quantities`, something which should improve identifiability.

In [ ]:
assess_identifiability(gwo; measured_quantities = [:M, :P])

What if we try another combination of meassured quantities? Here, the identifiability pattern changes, and can even improve.

In [ ]:
assess_identifiability(gwo; measured_quantities = [:M, :E])

In [ ]:
assess_identifiability(gwo; measured_quantities = [:E, :P])

This is an important utility of structural identifiaibltiy analysis. By telling us how identifiaibltiy depends on what we meassure, it is possible to better design experiments to maximise identifiability.